# Import Libraries

In [143]:
import boto3
import pandas as pd
import numpy as np
import sklearn
import sagemaker
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler

# Checking Up S3 Buckets and Locate Datasets

In [144]:
s3_resource = boto3.resource('s3')
bucket = s3_resource.Bucket('ads508-team1-sephora')

for items in bucket.objects.all():
    print(items.key)

athena/
athena/staging/
athena/staging/2db1a0d5-fa3b-49d6-92c3-f34da040eb33.txt
athena/staging/63402cb0-ca6f-40ed-bc68-e7980d01f348.txt
athena/staging/c427eba1-fadc-4d2a-ac8f-4b85a6e59415.txt
athena/staging/cfa0370a-881e-46f2-aa8e-0f8fa10ac191.txt
curated/
curated/products/
curated/products/products.parquet
curated/reviews/
curated/reviews/reviews.parquet
raw/
raw/products/
raw/products/product_info.csv
raw/reviews/
raw/reviews/reviews_0-250.csv
raw/reviews/reviews_1250-end.csv
raw/reviews/reviews_250-500.csv
raw/reviews/reviews_500-750.csv
raw/reviews/reviews_750-1250.csv


# Load the Product Info Dataset

In [145]:
# Define the S3 path
s3_path = 's3://ads508-team1-sephora/curated/products/products.parquet'

# Load the parquet directly into a DataFrame
df_products = pd.read_parquet(s3_path)

# Display the first few rows
df_products.head().T

,0,1,2,3,4
product_id,P473671,P473668,P473662,P473660,P473658
product_name,Fragrance Discovery Set,La Habana Eau de Parfum,Rainbow Bar Eau de Parfum,Kasbah Eau de Parfum,Purple Haze Eau de Parfum
brand_id,6342,6342,6342,6342,6342
brand_name,19-69,19-69,19-69,19-69,19-69
loves_count,6320,3827,3253,3018,2691
rating,3.6364,4.1538,4.25,4.4762,3.2308
reviews,11.0,13.0,16.0,21.0,13.0
size,None,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL
variation_type,None,Size + Concentration + Formulation,Size + Concentration + Formulation,Size + Concentration + Formulation,Size + Concentration + Formulation
variation_value,None,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL


In [146]:
# Product file shape
df_products.shape

(8494, 27)

# Project Will be Focus on Makeup Category

In [147]:
# Filter to only focus on the Makeup Category, since we only work on Makeup Category in our company
# Filter the dataframe to only include the Makeup category
df_products = df_products[df_products['primary_category'] == 'Makeup'].copy()

# Reset the index since we've removed rows from other categories
df_products = df_products.reset_index(drop=True)

# Quick check of the new dataset distribution
print(f"Filtered Shape: {df_products.shape}")
print("\nBreakdown of Makeup Sub-categories:")
print(df_products['secondary_category'].value_counts())

# Display the first few rows
df_products.head()


Filtered Shape: (2369, 27)

Breakdown of Makeup Sub-categories:
secondary_category
Eye                      711
Face                     659
Lip                      411
Brushes & Applicators    233
Cheek                    165
Nail                      52
Accessories               45
Mini Size                 37
Value & Gift Sets         36
Makeup Palettes           20
Name: count, dtype: int64


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P398965,Rose Lip Conditioner,7054,AERIN,20562,4.2600,527.0,0.34 oz/ 10 mL,Scent,1 Rose,...,1,0,0,"['Hydrating', 'Good for: Dryness']",Makeup,Lip,Lip Balm & Treatment,0,NaN,NaN
1,P503741,Lip Treatment Oil,7103,Ami Colé,22871,4.7053,95.0,0.15 oz / 4.5 ml,Color,Excellence,...,0,0,1,"['Vegan', 'High Shine Finish', 'Clean at Sepho...",Makeup,Lip,Lip Balm & Treatment,3,20.0,20.0
2,P503754,Skin-Enhancing Tinted Moisturizer,7103,Ami Colé,6596,4.8966,58.0,1 oz / 30 ml,Color,Rich 2,...,0,0,1,"['Vegan', 'Natural Finish', 'Clean at Sephora'...",Makeup,Face,Tinted Moisturizer,5,32.0,32.0
3,P503762,Lash-Amplifying Volumizing & Lengthening Mascara,7103,Ami Colé,5015,4.1842,38.0,0.3 oz / 9 ml,Color,Rich Black,...,0,0,1,"['Vegan', 'allure 2022 Best of Beauty Award Wi...",Makeup,Eye,Mascara,0,NaN,NaN
4,P503732,Skin Melt Talc-Free Loose Setting Powder,7103,Ami Colé,4978,4.5714,21.0,0. 29 oz / 8.6 g,Color,Rich / Deep,...,1,0,1,"['Vegan', 'Loose Powder Formula', 'Clean at Se...",Makeup,Face,Setting Spray & Powder,2,22.0,22.0


# Checking Out the Product Dataset Information after filtering

In [148]:
# Product File Information
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2369 entries, 0 to 2368
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          2369 non-null   object 
 1   product_name        2369 non-null   object 
 2   brand_id            2369 non-null   int64  
 3   brand_name          2369 non-null   object 
 4   loves_count         2369 non-null   int64  
 5   rating              2328 non-null   float64
 6   reviews             2328 non-null   float64
 7   size                1667 non-null   object 
 8   variation_type      1851 non-null   object 
 9   variation_value     1802 non-null   object 
 10  variation_desc      1186 non-null   object 
 11  ingredients         2033 non-null   object 
 12  price_usd           2369 non-null   float64
 13  value_price_usd     104 non-null    float64
 14  sale_price_usd      143 non-null    float64
 15  limited_edition     2369 non-null   int64  
 16  new   

In [149]:
# Profuct file description
df_products.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,2369,2369,P505461,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,2369,2360,Makeup Match Powder Brush,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand_id,2369.0,NaN,NaN,NaN,5120.040523,1683.804898,1063.0,3976.0,5879.0,6236.0,8000.0
brand_name,2369,115,SEPHORA COLLECTION,188,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loves_count,2369.0,NaN,NaN,NaN,54235.494301,100425.225843,0.0,8304.0,22387.0,53470.0,1401068.0
rating,2328.0,NaN,NaN,NaN,4.146845,0.513878,1.0,3.9286,4.2523,4.4915,5.0
reviews,2328.0,NaN,NaN,NaN,681.998282,1608.242011,1.0,44.0,209.0,625.0,21281.0
size,1667,978,1 oz/ 30 mL,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variation_type,1851,4,Color,1486,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variation_value,1802,1352,Black,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [150]:
# Checking missing data points
df_products.isnull().sum()

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                  41
reviews                 41
size                   702
variation_type         518
variation_value        567
variation_desc        1183
ingredients            336
price_usd                0
value_price_usd       2265
sale_price_usd        2226
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights             738
primary_category         0
secondary_category       0
tertiary_category      153
child_count              0
child_max_price       1069
child_min_price       1069
dtype: int64

In [151]:
# Checking missing data points percentage
df_products.isnull().mean()*100

product_id             0.000000
product_name           0.000000
brand_id               0.000000
brand_name             0.000000
loves_count            0.000000
rating                 1.730688
reviews                1.730688
size                  29.632756
variation_type        21.865766
variation_value       23.934149
variation_desc        49.936682
ingredients           14.183200
price_usd              0.000000
value_price_usd       95.609962
sale_price_usd        93.963698
limited_edition        0.000000
new                    0.000000
online_only            0.000000
out_of_stock           0.000000
sephora_exclusive      0.000000
highlights            31.152385
primary_category       0.000000
secondary_category     0.000000
tertiary_category      6.458421
child_count            0.000000
child_max_price       45.124525
child_min_price       45.124525
dtype: float64

# Imputation of Missing Value

## 1) Fill in mean rating for missing rating

In [152]:
# Fill in mean Rating for missing Rating
# Calculate the mean of the rating column
mean_rating = df_products['rating'].mean()

# Fill missing values in 'rating' with the mean
df_products['rating'] = df_products['rating'].fillna(mean_rating)

# Verify that there are no more missing values in 'rating'
print(f"Missing ratings remaining: {df_products['rating'].isnull().sum()}")

Missing ratings remaining: 0


## 2) Fill in zero for missing reviews

In [153]:
# Fill in zero for missing reviews
# Fill missing values in 'reviews' with 0
df_products['reviews'] = df_products['reviews'].fillna(0)

# Verify that there are no more missing values in 'reviews'
print(f"Missing reviews remaining: {df_products['reviews'].isnull().sum()}")

Missing reviews remaining: 0


## 3) Fill in NA for missing size, variation_type, variation_value, variation_desc, ingredients, highlights, secondary_category, tertiary_category

In [154]:
# Fill in NA for missing size, variation_type, variation_value, variation_desc, ingredients, highlights, secondary_category, tertiary_category
# List of columns to fill with "NA"
cols_to_fix = [
    'size', 'variation_type', 'variation_value', 
    'variation_desc', 'ingredients', 'highlights', 'secondary_category', 'tertiary_category'
]

# Fill missing values in these columns with "NA"
df_products[cols_to_fix] = df_products[cols_to_fix].fillna("NA")

# Verify that no missing values remain in these columns
print(df_products[cols_to_fix].isnull().sum())

size                  0
variation_type        0
variation_value       0
variation_desc        0
ingredients           0
highlights            0
secondary_category    0
tertiary_category     0
dtype: int64


## 4) Fill in price_usd for missing value in value_price_usd and sale_price_usd

In [155]:
# Fill in price_usd for missing value in value_price_usd and sale_price_usd
# Fill missing value_price_usd with the standard price_usd
df_products['value_price_usd'] = df_products['value_price_usd'].fillna(df_products['price_usd'])

# Fill missing sale_price_usd with the standard price_usd
df_products['sale_price_usd'] = df_products['sale_price_usd'].fillna(df_products['price_usd'])

# Verify that no missing values remain in these price columns
print(f"Missing value_price_usd: {df_products['value_price_usd'].isnull().sum()}")
print(f"Missing sale_price_usd: {df_products['sale_price_usd'].isnull().sum()}")

Missing value_price_usd: 0
Missing sale_price_usd: 0


## 5) Fill in zero for missing child_max_price and child_min_price

In [156]:
# Fill in zero for missing child_max_price and child_min_price
# Fill missing values for child price range columns with 0
df_products['child_max_price'] = df_products['child_max_price'].fillna(0)
df_products['child_min_price'] = df_products['child_min_price'].fillna(0)

# Verify the changes
print(f"Missing child_max_price: {df_products['child_max_price'].isnull().sum()}")
print(f"Missing child_min_price: {df_products['child_min_price'].isnull().sum()}")

Missing child_max_price: 0
Missing child_min_price: 0


## 6) Confirmation for no missing values

In [157]:
# Checking missing data points percentage
df_products.isnull().mean()*100

product_id            0.0
product_name          0.0
brand_id              0.0
brand_name            0.0
loves_count           0.0
rating                0.0
reviews               0.0
size                  0.0
variation_type        0.0
variation_value       0.0
variation_desc        0.0
ingredients           0.0
price_usd             0.0
value_price_usd       0.0
sale_price_usd        0.0
limited_edition       0.0
new                   0.0
online_only           0.0
out_of_stock          0.0
sephora_exclusive     0.0
highlights            0.0
primary_category      0.0
secondary_category    0.0
tertiary_category     0.0
child_count           0.0
child_max_price       0.0
child_min_price       0.0
dtype: float64

# Feature Engineering

## 1) Initial price feature engineering

In [158]:
# 1. Log transformation of price to handle skewness
df_products['price_log'] = np.log1p(df_products['price_usd'])

# 2. Identify if a product is actually on sale 
# (Comparing sale_price_usd to the original price_usd)
df_products['is_on_sale'] = (df_products['sale_price_usd'] < df_products['price_usd']).astype(int)

# 3. Calculate the discount percentage
# If no sale, this will result in 0.0
df_products['discount_pct'] = (df_products['price_usd'] - df_products['sale_price_usd']) / df_products['price_usd']

# 4. Calculate the value ratio (Value for money)
# (value_price_usd / price_usd)
df_products['value_ratio'] = df_products['value_price_usd'] / df_products['price_usd']

# Display the new features for the first few rows
df_products[['price_usd', 'price_log','sale_price_usd', 'is_on_sale', 'discount_pct', 'value_ratio']].head()

,price_usd,price_log,sale_price_usd,is_on_sale,discount_pct,value_ratio
0,35.0,3.583519,35.0,0,0.0,1.0
1,20.0,3.044522,20.0,0,0.0,1.0
2,32.0,3.496508,32.0,0,0.0,1.0
3,19.0,2.995732,19.0,0,0.0,1.0
4,22.0,3.135494,22.0,0,0.0,1.0


Log transformation of price_usd (price_log): Applied np.log1p to normalize the skewed price distribution and reduce the impact of extreme outliers (like the $1,900 items).

Price sensitivity & promotion tracking (is_on_sale & discount_pct): Created binary indicators and percentage calculations to capture whether a product is discounted relative to its original price.

Perceived value mapping (value_ratio): Calculated the ratio of value_price_usd to price_usd to quantify "value for money," especially for gift sets and bundles.

## 2) Price Bucketing (Business-friendly)

In [159]:
# Define price segments based on the distribution of price_usd
# New bins: 0-25, 25-50, 50-100, 100+
bins = [0, 25, 50, 100, np.inf]
labels = ['budget', 'low_mid', 'mid', 'premium']

# Create the price_bucket feature
df_products['price_bucket'] = pd.cut(df_products['price_usd'], bins=bins, labels=labels)

# Verify the distribution of the new buckets
print(df_products['price_bucket'].value_counts())

price_bucket
low_mid    1239
budget      895
mid         212
premium      23
Name: count, dtype: int64


Easier for Models & Business Interpretation: Converting exact prices into ranges reduces noise for machine learning algorithms and aligns with how retail stakeholders view product "tiers."

Hypothesis Testing: This allows us to move beyond individual price points and answer high-level business questions, such as: "Do premium products ($100–$200) maintain higher average ratings or 'loves' counts than budget items?"

## 3) Brand Features

In [160]:
# Identify the top 20 brands by product count
top_brands = df_products['brand_name'].value_counts().nlargest(20).index

# Group any brand not in the top 20 as 'Other' to reduce category cardinality
df_products['brand_group'] = df_products['brand_name'].apply(lambda x: x if x in top_brands else 'Other')

# Check the distribution of the new brand groups
print(df_products['brand_group'].value_counts())

brand_group
Other                      1082
SEPHORA COLLECTION          188
tarte                       100
Anastasia Beverly Hills      88
Charlotte Tilbury            76
Fenty Beauty by Rihanna      74
Hourglass                    67
MAKE UP FOR EVER             67
NARS                         59
CLINIQUE                     59
Dior                         56
HUDA BEAUTY                  54
Too Faced                    53
Urban Decay                  50
Benefit Cosmetics            49
PAT McGRATH LABS             49
Laura Mercier                47
Bobbi Brown                  39
Natasha Denona               38
Smashbox                     38
MILK MAKEUP                  36
Name: count, dtype: int64


Brand Categorization (brand_group): We identified the top 20 brands by product count and consolidated all remaining brands into a single "Other" category.

Solves High-Cardinality: Prevents "noise" in machine learning models by reducing hundreds of unique brand names—many with only 1 or 2 products—into a manageable set of features.

Captures Brand Reputation: Isolates the "Power Players" (e.g., SEPHORA COLLECTION, Clinique, Dior) where we have enough data to draw statistically significant conclusions about brand performance.

Dimensionality Reduction: Streamlines future visualizations and one-hot encoding, ensuring that our analysis (if we need to) focuses on the most influential market competitors rather than long-tail outliers.

## 4) One of the Outcome/Target - Popularity Features

In [161]:
# 1. Log transformation of loves_count
# Helps manage the massive range (0 to 1.4M+) and reduces skewness
df_products['log_loves'] = np.log1p(df_products['loves_count'])

# 2. Loves per Price (Value Sentiment)
# Measures how much 'love' a product gets per dollar spent
df_products['loves_per_price'] = df_products['loves_count'] / df_products['price_usd']

# 3. Popularity Score (Interaction weighted by Rating)
# A simple way to combine volume (reviews) and sentiment (rating)
df_products['popularity_index'] = df_products['reviews'] * df_products['rating']

# Display the results for high-interaction products
df_products[['product_name', 'loves_count', 'log_loves', 'loves_per_price', 'popularity_index']].sort_values(by='loves_count', ascending=False).head()

,product_name,loves_count,log_loves,loves_per_price,popularity_index
1645,Soft Pinch Liquid Blush,1401068,14.152746,60916.000000,21466.9948
1404,Radiant Creamy Concealer,1153594,13.958394,36049.812500,55517.1960
1742,Cream Lip Stain Liquid Lipstick,1029051,13.844149,68603.400000,48000.6311
550,Gloss Bomb Universal Lip Luminizer,968317,13.783316,46110.333333,56258.8552
551,Pro Filt’r Soft Matte Longwear Liquid Foundation,856497,13.660607,21412.425000,68342.8860


Popularity & Engagement Metrics (log_loves, loves_per_price, popularity_index): We engineered several features to quantify product "hype" and consumer sentiment.

Captures Customer Demand: By combining loves_count and reviews, we create a proxy for market demand that distinguishes between products people simply "wishlist" versus those they actually purchase and evaluate.

Addresses Popularity Bias: Applying a log transformation to loves_count (which ranges from 0 to over 1.4M) prevents "viral" outliers—like the Soft Pinch Liquid Blush—from skewing the model's ability to learn from standard products.

Identifies "Cult Favorites": The popularity_index (Reviews × Rating) helps the model prioritize products with high-volume, high-quality feedback, effectively filtering out 5-star items that only have a single review.

## 5) Ingredient Features (Text → Structured)

In [162]:
# 1. Calculate the number of ingredients
# We strip the brackets and quotes from the string representation before splitting
df_products['num_ingredients'] = df_products['ingredients'].apply(
    lambda x: len(str(x).replace('[', '').replace(']', '').split(',')) if x != "NA" else 0
)

# 2. Extract top 100 ingredient features using Bag-of-Words (CountVectorizer)
vectorizer = CountVectorizer(max_features=100, stop_words='english')

# Ensure we use the string version of the ingredients for the vectorizer
ingredient_text = df_products['ingredients'].astype(str)
ingredient_matrix = vectorizer.fit_transform(ingredient_text)

# 3. Convert the matrix into a DataFrame and join it back to df_products
ingredient_df = pd.DataFrame(
    ingredient_matrix.toarray(), 
    columns=[f"ingred_{name}" for name in vectorizer.get_feature_names_out()]
)

# Reset index to ensure proper alignment during concatenation
df_products = pd.concat([df_products.reset_index(drop=True), ingredient_df], axis=1)

# Verify the new columns
print(f"Total columns after ingredient engineering: {len(df_products.columns)}")
df_products[['product_name', 'num_ingredients']].head()

Total columns after ingredient engineering: 137


,product_name,num_ingredients
0,Rose Lip Conditioner,3
1,Lip Treatment Oil,24
2,Skin-Enhancing Tinted Moisturizer,25
3,Lash-Amplifying Volumizing & Lengthening Mascara,32
4,Skin Melt Talc-Free Loose Setting Powder,19


Ingredient Analysis & Vectorization (num_ingredients, ingred_...): We transformed the raw, unstructured ingredient lists into numerical features using a combination of string parsing and natural language processing (NLP).

Ingredient Complexity as a Quality Signal: The num_ingredients count serves as a proxy for formula sophistication; in the beauty industry, a higher count or specific "hero" ingredients often correlate with premium pricing and perceived product efficacy.

Enables NLP-Based Modeling: By using CountVectorizer to extract the top 100 most frequent ingredients, we've enabled the model to identify "chemical signatures." This allows the algorithm to detect if specific components—like Hyaluronic Acid or Squalane—are statistically linked to higher customer satisfaction.

Feature Expansion: This step converted a single text column into 101 structured data points, allowing the machine learning model to "read" the label and weigh the impact of formula composition on a product's overall rating.

# Feature Transformation

## 1) Normalize numerical features

In [163]:
# 1. Identify truly continuous/numerical columns
# We filter for columns that have a wide range of values 
# and exclude the 0/1 binary columns explicitly.

cols_to_scale = []
for col in df_products.select_dtypes(include=['number']).columns:
    # Skip IDs
    if col == 'brand_id':
        continue
    
    # Check if the column is binary (only has 2 unique values, e.g., 0 and 1)
    if df_products[col].nunique() <= 2:
        continue
        
    # Optional: Skip columns that start with 'ingred_' if you want 
    # to keep ingredient counts as-is
    if col.startswith('ingred_'):
        continue
        
    cols_to_scale.append(col)

print("Refined list of columns to scale:")
print(cols_to_scale)

# 1. Select the numerical columns that need scaling
# We exclude binary/dummy columns (0/1) as they are already bounded
cols_to_scale = [
    'price_usd', 'value_price_usd', 
    'sale_price_usd', 'child_count', 'child_max_price', 'child_min_price', 
    'price_log', 'discount_pct', 'value_ratio', 'num_ingredients'
]

# 2. Initialize and apply the StandardScaler
scaler = StandardScaler()
df_products[cols_to_scale] = scaler.fit_transform(df_products[cols_to_scale])

# 3. Verify the transformation (Means should be approx 0 and Std should be 1)
df_products[cols_to_scale].describe().round(2).T

Refined list of columns to scale:
['loves_count', 'rating', 'reviews', 'price_usd', 'value_price_usd', 'sale_price_usd', 'child_count', 'child_max_price', 'child_min_price', 'price_log', 'discount_pct', 'value_ratio', 'log_loves', 'loves_per_price', 'popularity_index', 'num_ingredients']


,count,mean,std,min,25%,50%,75%,max
price_usd,2369.0,0.0,1.0,-1.60,-0.58,-0.20,0.33,15.40
value_price_usd,2369.0,-0.0,1.0,-1.64,-0.57,-0.23,0.30,13.83
sale_price_usd,2369.0,-0.0,1.0,-1.57,-0.53,-0.21,0.32,15.24
child_count,2369.0,-0.0,1.0,-0.51,-0.51,-0.40,0.03,10.86
child_max_price,2369.0,0.0,1.0,-0.90,-0.90,-0.09,0.66,10.15
child_min_price,2369.0,0.0,1.0,-0.88,-0.88,-0.15,0.64,10.64
price_log,2369.0,0.0,1.0,-4.38,-0.59,-0.02,0.61,5.11
discount_pct,2369.0,0.0,1.0,-0.25,-0.25,-0.25,-0.25,6.82
value_ratio,2369.0,-0.0,1.0,-5.70,-0.16,-0.16,-0.16,16.21
num_ingredients,2369.0,-0.0,1.0,-0.73,-0.41,-0.18,0.06,14.64


Feature Standardization (StandardScaler): we applied $z$-score normalization to all continuous numerical variables (e.g., price_usd, loves_count, num_ingredients) to ensure they share a common scale with a mean of 0 and a standard deviation of 1.

Ensures Model Fairness: machine learning algorithms (like Linear Regression or KNN) often misinterpret large raw numbers—such as a loves_count of 1,000,000—as being mathematically more "important" than a rating of 4.5. Scaling prevents this bias by putting all features on a level playing field.

Improves Convergence: for gradient-based models (like Neural Networks or XGBoost), having features on the same scale allows the model to find the optimal solution much faster and more reliably.

Handles Distribution Variance: by standardizing engineered features like popularity_index and loves_per_price, we make the relative "success" of a product comparable across the entire dataset, regardless of the original unit of measurement.

## 2) Category features transformation

In [164]:
# Get a list of all categorical column names
categorical_cols_list = df_products.select_dtypes(include=['object', 'category']).columns.tolist()

# Print the count and the names
print(f"Total categorical columns: {len(categorical_cols_list)}")
print(categorical_cols_list)

# 1. Define all categorical features that provide structural meaning
categorical_cols = [ 'secondary_category','tertiary_category', 
                    'price_bucket', 'brand_group'
]

# 2. Apply One-Hot Encoding across all selected features
# We include brand_group and price_bucket to capture brand equity and market tier signals
df_products = pd.get_dummies(
    df_products, 
    columns=categorical_cols, 
    drop_first=True,
    dtype=int
)

Total categorical columns: 14
['product_id', 'product_name', 'brand_name', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'price_bucket', 'brand_group']


Categorical Encoding (like secondary_category): We converted the text-based category labels into a series of binary (0 or 1) columns using One-Hot Encoding.

Category-Specific Success Patterns: Product categories strongly influence performance metrics; for example, Skincare products often have higher price points and repeat purchase rates compared to Makeup, which is more trend-driven.

Eliminating the "Dummy Variable Trap": By using drop_first=True, we ensured mathematical independence between our new columns, preventing multicollinearity which can skew model coefficients.

Machine Learning Readiness: Since algorithms cannot "read" text, this transformation translates the product's department into a numerical format that allows the model to calculate the specific "weight" or impact of being in a high-growth category like Fragrance versus Bath & Body.

# Final Product Dataset

In [165]:
# We drop raw text and ID columns to keep the dataframe strictly numeric/encoded
cols_to_drop = [
    'product_id', 'product_name', 'brand_id', 'brand_name',
    'loves_count', 'rating', 'reviews', 'log_loves', 'loves_per_price', 'popularity_index',
    'size', 'variation_type', 'variation_value', 
    'variation_desc', 'ingredients', 'highlights' 
]

df_product_final = df_products.drop(columns=cols_to_drop)

# Verify the final shape (should be significantly wider now)
print(f"Final Product Features: {df_product_final.shape[1]}")
df_product_final.head().T

Final Product Features: 190


,0,1,2,3,4
price_usd,0.120179,-0.683872,-0.040631,-0.737475,-0.576665
value_price_usd,0.056327,-0.668544,-0.088647,-0.716869,-0.571894
sale_price_usd,0.15904,-0.634933,0.000245,-0.687865,-0.52907
limited_edition,0,0,0,0,0
new,0,0,0,0,0
...,...,...,...,...,...
brand_group_SEPHORA COLLECTION,0,0,0,0,0
brand_group_Smashbox,0,0,0,0,0
brand_group_Too Faced,0,0,0,0,0
brand_group_Urban Decay,0,0,0,0,0
